In [1]:
import pandas as pd

In [2]:
df_query = pd.read_csv('input/mvp_remapping_db_results_disease_type.csv',na_filter=False)
len(df_query)

336

In [3]:
df_query

,model_name,disease_name,question_id,value,count_of_values,case_count
0,General,Meningococcal disease (Neisseria meningitidis),CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,1,3
1,General,Pertussis,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,73,2730
2,General,Streptococcus pneumoniae,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,97,612
3,General,Measles (rubeola),CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,30,3854
4,General,Diphtheria Non Respiratory Toxogenic,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,2,101
...,...,...,...,...,...,...
331,General,Malaria,INVESTIGATION_STATUS,ASSIGNED,4,44
332,General,MDRO,INVESTIGATION_STATUS,ASSIGNED,2,2
333,General,OD/Standby Calls,INVESTIGATION_STATUS,COMPLETE,66,93
334,General,Typhoid fever,INVESTIGATION_STATUS,ASSIGNED,1,1


In [4]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:]==answer):
        return question[:start].strip('_')
    else:
        return question

In [5]:
df_query.loc[:,'parent_question_id']  = [ join_all_except_last(q,a) for q,a in zip(df_query.loc[:,'question_id'],df_query.loc[:,'value'])]

In [6]:
df_query

,model_name,disease_name,question_id,value,count_of_values,case_count,parent_question_id
0,General,Meningococcal disease (Neisseria meningitidis),CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,1,3,CASE_INVESTIGATION_STATUS
1,General,Pertussis,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,73,2730,CASE_INVESTIGATION_STATUS
2,General,Streptococcus pneumoniae,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,97,612,CASE_INVESTIGATION_STATUS
3,General,Measles (rubeola),CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,30,3854,CASE_INVESTIGATION_STATUS
4,General,Diphtheria Non Respiratory Toxogenic,CASE_INVESTIGATION_STATUS_ASSIGNED,ASSIGNED,2,101,CASE_INVESTIGATION_STATUS
...,...,...,...,...,...,...,...
331,General,Malaria,INVESTIGATION_STATUS,ASSIGNED,4,44,INVESTIGATION_STATUS
332,General,MDRO,INVESTIGATION_STATUS,ASSIGNED,2,2,INVESTIGATION_STATUS
333,General,OD/Standby Calls,INVESTIGATION_STATUS,COMPLETE,66,93,INVESTIGATION_STATUS
334,General,Typhoid fever,INVESTIGATION_STATUS,ASSIGNED,1,1,INVESTIGATION_STATUS


In [7]:
# Group by parent_question_id and model_name to get the num of values reported
df_group = df_query.groupby(by=['parent_question_id','model_name','disease_name','case_count'])['count_of_values'].sum().reset_index(name='count_of_values')

In [8]:
print(df_group)

       parent_question_id model_name            disease_name  case_count  \
0                            General      Angiostrongyliasis          59   
1                            General            Dengue fever          74   
2                            General              Filariasis           8   
3                            General       Measles (rubeola)        3854   
4                            General        OD/Standby Calls          93   
..                    ...        ...                     ...         ...   
178  INVESTIGATION_STATUS  Hepatitis    Hepatitis B, contact          51   
179  INVESTIGATION_STATUS  Hepatitis  Hepatitis B, perinatal          85   
180  INVESTIGATION_STATUS  Hepatitis  Hepatitis B, pregnancy          84   
181  INVESTIGATION_STATUS  Hepatitis      Hepatitis C, acute          55   
182  INVESTIGATION_STATUS  Hepatitis    Hepatitis C, chronic        1773   

     count_of_values  
0                  3  
1                 15  
2                 

In [9]:
df_group.loc[:,'fill_rate'] = [round(c*100.0/t,3) for c,t in zip(df_group.loc[:,'count_of_values'],df_group.loc[:,'case_count'])]

In [ ]:
df_group 

,parent_question_id,model_name,disease_name,case_count,count_of_values,fill_rate
0,,General,Angiostrongyliasis,59,3,5.085
1,,General,Dengue fever,74,15,20.270
2,,General,Filariasis,8,1,12.500
3,,General,Measles (rubeola),3854,2,0.052
4,,General,OD/Standby Calls,93,1,1.075
...,...,...,...,...,...,...
178,INVESTIGATION_STATUS,Hepatitis,"Hepatitis B, contact",51,18,35.294
179,INVESTIGATION_STATUS,Hepatitis,"Hepatitis B, perinatal",85,86,101.176
180,INVESTIGATION_STATUS,Hepatitis,"Hepatitis B, pregnancy",84,95,113.095
181,INVESTIGATION_STATUS,Hepatitis,"Hepatitis C, acute",55,1,1.818


In [12]:
df_group.to_csv('output/fill_rate_disease_type.csv',index=False)